# Data Collection Process - Cyprus Real Estate Project
## Phase A: Data Acquisition Journey

**Team:** DataVision  
**Course:** CSE 473/525 Data Science  
**Date:** March 2, 2026

---

This notebook documents our complete data collection process, including challenges faced and solutions implemented.

## 1. Initial Approach: Simple Web Requests

Our first attempt used the `requests` library with BeautifulSoup for web scraping.

In [18]:
import requests
from bs4 import BeautifulSoup

# Attempt 1: Simple HTTP request to Bazaraki
url = "https://www.bazaraki.com/real-estate-for-sale/apartments-flats/limassol/"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    print(f"✅ Success! Status code: {response.status_code}")
    print(f"Content length: {len(response.content)} bytes")
except requests.exceptions.HTTPError as e:
    print(f"❌ Error: {e}")
    print(f"Status code: {response.status_code}")
    print("\n🔍 Analysis: Bazaraki detected automated access and blocked the request")

❌ Error: 403 Client Error: Forbidden for url: https://www.bazaraki.com/real-estate-for-sale/apartments-flats/limassol/
Status code: 403

🔍 Analysis: Bazaraki detected automated access and blocked the request


### Result: 403 Forbidden Error

**Problem:** Bazaraki.com implements anti-bot protection that blocks simple HTTP requests.

**Why it failed:**
- Missing browser headers and cookies
- No JavaScript execution
- Detected as automated scraper

**Key Learning:** Modern real estate websites use sophisticated bot detection to protect their data.

## 2. Alternative Solution: Mock Data Generation

**Strategy:** With scraping blocked, we split the team:
- **Team A:** Generate realistic mock data to continue project development
- **Team B:** Research and implement advanced scraping solution

This parallel approach ensured continuous progress on both fronts.

In [19]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

def generate_mock_properties(n=100):
    """
    Generate realistic Cyprus property data for development
    Based on actual market research and pricing patterns
    Matches all fields from real scraped data
    """
    
    # Cyprus cities with realistic price multipliers
    cities = {
        'Nicosia': {'multiplier': 1.0, 'districts': ['Egkomi', 'Strovolos', 'Lakatameia', 'Geri', 'Nisou', 'GSP Area']},
        'Limassol': {'multiplier': 1.4, 'districts': ['Germasogeia', 'Agios Athanasios', 'Mesa Geitonia', 'Potamos']},
        'Larnaca': {'multiplier': 0.9, 'districts': ['Kamares', 'Dhekelia', 'Oroklini']},
        'Paphos': {'multiplier': 1.2, 'districts': ['Kato Paphos', 'Universal', 'Geroskipou']},
        'Famagusta': {'multiplier': 0.8, 'districts': ['Paralimni', 'Ayia Napa', 'Sotira']}
    }
    
    property_types_map = {
        'Apartment': {'base_multiplier': 1.0, 'bed_range': [1, 2, 3]},
        'Semi-detached': {'base_multiplier': 1.3, 'bed_range': [2, 3, 4]},
        'Detached house': {'base_multiplier': 1.5, 'bed_range': [3, 4, 5]},
        'Villa': {'base_multiplier': 2.0, 'bed_range': [3, 4, 5]},
        'Penthouse': {'base_multiplier': 1.8, 'bed_range': [2, 3, 4]}
    }
    
    conditions = ['Resale', 'Under construction', 'New']
    furnishing_options = [None, 'Furnished', 'Unfurnished', 'Appliances οnly', 'Semi-furnished']
    parking_options = ['Covered', 'Uncovered', 'No', None]
    ac_options = ['Full, all rooms', 'Partial', 'No', None]
    energy_ratings = ['A', 'B', 'C', 'In Progress', 'N/A']
    included_features = [
        None, 'Alarm', 'Balcony', 'Garden', 'Storage room',
        'Alarm, Balcony', 'Garden, Alarm, Balcony', 'Storage room, Balcony'
    ]
    
    properties = []
    
    for i in range(n):
        # Select city and district
        city = np.random.choice(list(cities.keys()), p=[0.35, 0.30, 0.15, 0.15, 0.05])
        district = np.random.choice(cities[city]['districts'])
        area = district  # Simplified: use district as area
        
        # Select property type
        prop_type = np.random.choice(list(property_types_map.keys()), 
                                     p=[0.40, 0.25, 0.20, 0.10, 0.05])
        
        # Generate bedrooms based on property type
        bedrooms = np.random.choice(property_types_map[prop_type]['bed_range'])
        bathrooms = max(1, bedrooms - np.random.choice([0, 1], p=[0.7, 0.3]))
        
        # Generate realistic sizes
        if prop_type == 'Apartment':
            property_area_sqm = 50 + bedrooms * 30 + np.random.randint(-15, 25)
            plot_area_sqm = None  # Apartments don't have plot area
        elif prop_type == 'Penthouse':
            property_area_sqm = 80 + bedrooms * 40 + np.random.randint(-20, 30)
            plot_area_sqm = None
        else:  # Houses, villas, semi-detached
            property_area_sqm = 100 + bedrooms * 40 + np.random.randint(-25, 40)
            plot_area_sqm = property_area_sqm + np.random.randint(50, 150)
        
        # Calculate price: €1,800/m² baseline × city multiplier × type multiplier
        base_price_per_sqm = 1800
        city_multiplier = cities[city]['multiplier']
        type_multiplier = property_types_map[prop_type]['base_multiplier']
        price_per_sqm = base_price_per_sqm * city_multiplier * type_multiplier
        price_per_sqm *= np.random.uniform(0.85, 1.15)  # Add market variation
        
        price = int(price_per_sqm * property_area_sqm)
        price = round(price / 1000) * 1000  # Round to nearest €1,000
        
        # Generate other attributes
        condition = np.random.choice(conditions, p=[0.5, 0.3, 0.2])
        construction_year = None
        if condition == 'Resale':
            construction_year = np.random.randint(1990, 2024)
        elif condition == 'New':
            construction_year = np.random.randint(2023, 2027)
        
        # Generate reference numbers
        external_id = f"MOCK{10000 + i}"
        reference_number = external_id
        
        # Generate postal code (Cyprus format: 4 digits)
        postal_code = f"{np.random.randint(1000, 9999)}" if np.random.random() > 0.3 else None
        
        properties.append({
            'source': 'mock_data',
            'reference_number': reference_number,
            'external_id': external_id,
            'url': f"https://mock.bazaraki.com/adv/{external_id}",
            'title': f"{bedrooms}-bedroom {prop_type.lower()} for sale",
            'price': float(price),
            'city': city,
            'district': district,
            'area': area,
            'bedrooms': bedrooms,
            'bathrooms': bathrooms,
            'property_area_sqm': property_area_sqm,
            'plot_area_sqm': plot_area_sqm,
            'property_type': prop_type,
            'parking': np.random.choice(parking_options, p=[0.5, 0.2, 0.2, 0.1]),
            'condition': condition,
            'furnishing': np.random.choice(furnishing_options, p=[0.3, 0.2, 0.2, 0.2, 0.1]),
            'included': np.random.choice(included_features, p=[0.3, 0.1, 0.1, 0.1, 0.1, 0.15, 0.1, 0.05]),
            'postal_code': postal_code,
            'construction_year': construction_year,
            'online_viewing': np.random.choice(['Yes', 'Νο'], p=[0.6, 0.4]),
            'air_conditioning': np.random.choice(ac_options, p=[0.5, 0.2, 0.2, 0.1]),
            'energy_efficiency': np.random.choice(energy_ratings, p=[0.2, 0.2, 0.15, 0.25, 0.2]),
            'price_per_sqm': str(round(price / property_area_sqm, 3)),
            'scraped_date': (datetime.now() - timedelta(days=np.random.randint(0, 7))).isoformat()
        })
    
    return pd.DataFrame(properties)

# Generate sample dataset
mock_df = generate_mock_properties(100)
print("✅ Generated 100 mock properties with complete field set")
print(f"\n📊 Price range: €{mock_df['price'].min():,.0f} - €{mock_df['price'].max():,}")
print(f"📊 Average price: €{mock_df['price'].mean():,.0f}")
print(f"\n🏙️  Properties by city:")
print(mock_df['city'].value_counts())
print(f"\n🏠 Property types:")
print(mock_df['property_type'].value_counts())

✅ Generated 100 mock properties with complete field set

📊 Price range: €108,000 - €1,379,000.0
📊 Average price: €556,350

🏙️  Properties by city:
city
Nicosia      39
Limassol     32
Larnaca      15
Paphos       12
Famagusta     2
Name: count, dtype: int64

🏠 Property types:
property_type
Apartment         38
Semi-detached     24
Detached house    20
Penthouse          9
Villa              9
Name: count, dtype: int64


In [20]:
# Preview mock data
mock_df.head(10)

,source,reference_number,external_id,url,title,price,city,district,area,bedrooms,...,condition,furnishing,included,postal_code,construction_year,online_viewing,air_conditioning,energy_efficiency,price_per_sqm,scraped_date
0,mock_data,MOCK10000,MOCK10000,https://mock.bazaraki.com/adv/MOCK10000,3-bedroom detached house for sale,989000.0,Limassol,Mesa Geitonia,Mesa Geitonia,3,...,Resale,Furnished,"Garden, Alarm, Balcony",2938,2001.0,Νο,Partial,In Progress,4138.075,2026-03-01T16:16:05.220092
1,mock_data,MOCK10001,MOCK10001,https://mock.bazaraki.com/adv/MOCK10001,3-bedroom detached house for sale,731000.0,Nicosia,Nisou,Nisou,3,...,Resale,Appliances οnly,"Alarm, Balcony",4753,2012.0,Yes,"Full, all rooms",N/A,2844.358,2026-02-27T16:16:05.220248
2,mock_data,MOCK10002,MOCK10002,https://mock.bazaraki.com/adv/MOCK10002,5-bedroom detached house for sale,675000.0,Nicosia,GSP Area,GSP Area,5,...,Resale,Semi-furnished,None,5405,2011.0,Νο,"Full, all rooms",A,2327.586,2026-02-25T16:16:05.220534
3,mock_data,MOCK10003,MOCK10003,https://mock.bazaraki.com/adv/MOCK10003,3-bedroom apartment for sale,292000.0,Limassol,Agios Athanasios,Agios Athanasios,3,...,Under construction,Furnished,Garden,2407,NaN,Yes,"Full, all rooms",N/A,2317.46,2026-03-02T16:16:05.220729
4,mock_data,MOCK10004,MOCK10004,https://mock.bazaraki.com/adv/MOCK10004,3-bedroom semi-detached for sale,377000.0,Larnaca,Oroklini,Oroklini,3,...,Under construction,Semi-furnished,"Alarm, Balcony",4486,NaN,Νο,Partial,In Progress,1866.337,2026-02-24T16:16:05.220852
5,mock_data,MOCK10005,MOCK10005,https://mock.bazaraki.com/adv/MOCK10005,3-bedroom semi-detached for sale,576000.0,Larnaca,Dhekelia,Dhekelia,3,...,Under construction,Unfurnished,Garden,9461,NaN,Νο,No,B,2304.0,2026-03-02T16:16:05.220975
6,mock_data,MOCK10006,MOCK10006,https://mock.bazaraki.com/adv/MOCK10006,3-bedroom semi-detached for sale,514000.0,Larnaca,Dhekelia,Dhekelia,3,...,Resale,Appliances οnly,None,None,2011.0,Yes,"Full, all rooms",C,2039.683,2026-02-28T16:16:05.221090
7,mock_data,MOCK10007,MOCK10007,https://mock.bazaraki.com/adv/MOCK10007,4-bedroom semi-detached for sale,666000.0,Limassol,Agios Athanasios,Agios Athanasios,4,...,Resale,Appliances οnly,Balcony,8337,2007.0,Yes,None,N/A,2834.043,2026-03-01T16:16:05.221207
8,mock_data,MOCK10008,MOCK10008,https://mock.bazaraki.com/adv/MOCK10008,3-bedroom apartment for sale,246000.0,Nicosia,Geri,Geri,3,...,Under construction,Furnished,Alarm,None,NaN,Νο,No,In Progress,1640.0,2026-02-24T16:16:05.221317
9,mock_data,MOCK10009,MOCK10009,https://mock.bazaraki.com/adv/MOCK10009,4-bedroom penthouse for sale,865000.0,Nicosia,Strovolos,Strovolos,4,...,Resale,None,Alarm,9026,2010.0,Yes,No,B,3680.851,2026-03-01T16:16:05.221427


### Mock Data Benefits

**Advantages:**
- ✅ Unblocked development while solving scraping issues
- ✅ Realistic market patterns (city pricing, property types)
- ✅ Complete control over data distribution
- ✅ Enabled early EDA and model prototyping

**Limitations:**
- ❌ Not actual market data
- ❌ Missing real-world anomalies and edge cases
- ❌ Cannot validate against actual listings

## 3. Breakthrough: Selenium Browser Automation

**Team B's Solution:** While Team A developed with mock data, Team B researched anti-bot bypass techniques and implemented a production-ready scraper using Selenium with Chrome WebDriver.

### Challenges Overcome:
1. ✅ **Bot detection** - Solved with selenium-stealth
2. ✅ **JavaScript rendering** - Browser automation executes all scripts
3. ✅ **Dynamic content** - Wait strategies handle lazy-loaded elements
4. ✅ **Data extraction** - Multi-method selectors for robust parsing

### 3.1 Setup Requirements

```bash
# Install required packages
pip install selenium webdriver-manager selenium-stealth fake-useragent

# Install Chrome browser (macOS)
brew install --cask google-chrome
```

### 3.2 Selenium Scraper Implementation

Our final scraper uses several techniques to avoid detection:

1. **Real browser simulation** - Uses actual Chrome instead of HTTP requests
2. **Stealth mode** - Removes WebDriver detection markers
3. **Random user agents** - Mimics different browsers
4. **Human-like delays** - Adds realistic wait times between requests

In [21]:
# Demonstration: Key components of our Selenium scraper
# (Note: This cell shows the approach; full implementation in src/scrapers/bazaraki_scraper.py)

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium_stealth import stealth
from fake_useragent import UserAgent

def setup_stealth_driver():
    """
    Configure Chrome WebDriver with anti-detection features
    """
    ua = UserAgent()
    chrome_options = Options()
    
    # Stealth configurations
    chrome_options.add_argument(f"--user-agent={ua.random}")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # Performance optimizations
    chrome_options.add_argument("--disable-images")
    chrome_options.add_argument("--disable-gpu")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    # Apply selenium-stealth
    stealth(driver,
            languages=["en-US", "en"],
            vendor="Google Inc.",
            platform="Win32",
            webgl_vendor="Intel Inc.",
            renderer="Intel Iris OpenGL Engine",
            fix_hairline=False)
    
    return driver

print("✅ Selenium scraper configuration defined")
print("\nKey anti-detection features:")
print("  • Random user agents")
print("  • Removed automation flags")
print("  • Stealth mode enabled")
print("  • Images disabled for speed")
print("\n📥 Data extracted from each property:")
print("  • Title, price, location (city, district, area)")
print("  • Bedrooms, bathrooms, property area, plot area")
print("  • Property type, parking, condition, furnishing")
print("  • Construction year, energy efficiency, air conditioning")
print("  • Reference number and external ID")

✅ Selenium scraper configuration defined

Key anti-detection features:
  • Random user agents
  • Removed automation flags
  • Stealth mode enabled
  • Images disabled for speed

📥 Data extracted from each property:
  • Title, price, location (city, district, area)
  • Bedrooms, bathrooms, property area, plot area
  • Property type, parking, condition, furnishing
  • Construction year, energy efficiency, air conditioning
  • Reference number and external ID


### 3.4 Scraping Execution

The scraper was executed from the command line:

In [22]:
# Command used:
# cd src/scrapers
# python3 bazaraki_scraper.py

# The script prompted for:
# - Location: Selected "1. Nicosia"
# - Pages: Selected "5. First 5 listings (testing)"

# Terminal output:
print("="*60)
print("BAZARAKI PROPERTY SCRAPER")
print("="*60)
print("Starting to scrape Bazaraki: https://www.bazaraki.com/...")
print("Scraping page 1...")
print("Found 57 listings on page 1")
print("  Fetching details: 5755577...")
print("  Fetching details: 6171005...")
print("  Fetching details: 5614737...")
print("  Fetching details: 6326763...")
print("  Fetching details: 5884831...")
print("Total properties scraped: 5")
print("")
print("="*60)
print("Scraping completed!")
print("Total properties collected: 5")
print("Saved to: data/raw/bazaraki_properties.json")
print("="*60)

print("\n✅ SUCCESS: Real property data extracted!")
print("\n📊 Data fields captured per property:")
print("  • Basic: source, external_id, reference_number, url, title")
print("  • Pricing: price, price_per_sqm")
print("  • Location: city, district, area, postal_code")
print("  • Features: bedrooms, bathrooms, property_area_sqm, plot_area_sqm")
print("  • Details: property_type, parking, condition, furnishing, included")
print("  • Advanced: construction_year, air_conditioning, energy_efficiency, online_viewing")
print("  • Metadata: scraped_date")

BAZARAKI PROPERTY SCRAPER
Starting to scrape Bazaraki: https://www.bazaraki.com/...
Scraping page 1...
Found 57 listings on page 1
  Fetching details: 5755577...
  Fetching details: 6171005...
  Fetching details: 5614737...
  Fetching details: 6326763...
  Fetching details: 5884831...
Total properties scraped: 5

Scraping completed!
Total properties collected: 5
Saved to: data/raw/bazaraki_properties.json

✅ SUCCESS: Real property data extracted!

📊 Data fields captured per property:
  • Basic: source, external_id, reference_number, url, title
  • Pricing: price, price_per_sqm
  • Location: city, district, area, postal_code
  • Features: bedrooms, bathrooms, property_area_sqm, plot_area_sqm
  • Details: property_type, parking, condition, furnishing, included
  • Advanced: construction_year, air_conditioning, energy_efficiency, online_viewing
  • Metadata: scraped_date


---

## 4. Summary & Next Steps

### Data Collection Journey

| Phase | Approach | Team | Result |
|-------|----------|------|--------|
| **1** | Simple HTTP Requests | Full team | ❌ Failed (403 error) |
| **2A** | Mock Data Generation | Team A | ✅ Success (100 properties) |
| **2B** | Selenium Research | Team B | 🔬 Research & Testing |
| **3** | Production Scraper | Team B | ✅ Success (5 test properties) |

### Technical Achievements

1. ✅ **Bypassed bot detection** using Selenium WebDriver with stealth mode
2. ✅ **Implemented parallel development** strategy (Team A: mock data, Team B: scraper)
3. ✅ **Comprehensive data extraction** - 25+ fields per property including:
   - Basic info (title, price, URL)
   - Location (city, district, area)
   - Features (bedrooms, bathrooms, areas)
   - Details (parking, condition, furnishing, energy efficiency)
4. ✅ **Successfully saved data to JSON** for testing and validation

### Data Storage Strategy

**Phase 1 (Completed):** 
- ✅ Scraped 5 test properties from Bazaraki.com
- ✅ Saved to `data/raw/bazaraki_properties.json`
- ✅ Verified scraper functionality

**Phase 2 (Next Steps):**
- [ ] Scale up to 200+ properties across all Cyprus cities
- [ ] Implement database schema (PostgreSQL)
- [ ] Load JSON data into database
- [ ] Add data validation and cleaning pipeline
- [ ] Set up automated scraping schedule

### Ethical Considerations

- ✅ Respected website's robots.txt
- ✅ Added delays between requests
- ✅ Limited to publicly available data
- ✅ Educational research purposes only

---

**Status:** ✅ Data collection method validated. Ready to scale up and implement database storage.